# Exploration des ventes immobilières à La Réunion (DVF 2024)

Données **DVF** (Demandes de valeurs foncières) pour le département 974, année 2024 : toutes les ventes immobilières enregistrées par les notaires.

Objectif : comprendre la structure du fichier, repérer ses pièges et calculer un premier prix médian au m² par commune.

## Combien de lignes et de colonnes ? Quels types de données ?

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/974_2024.csv.gz", low_memory=False)
df.shape, df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19251 entries, 0 to 19250
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_mutation                   19251 non-null  str    
 1   date_mutation                 19251 non-null  str    
 2   numero_disposition            19251 non-null  int64  
 3   nature_mutation               19251 non-null  str    
 4   valeur_fonciere               19203 non-null  float64
 5   adresse_numero                16673 non-null  float64
 6   adresse_suffixe               1766 non-null   str    
 7   adresse_nom_voie              19223 non-null  str    
 8   adresse_code_voie             19223 non-null  str    
 9   code_postal                   19223 non-null  float64
 10  code_commune                  19251 non-null  int64  
 11  nom_commune                   19251 non-null  str    
 12  code_departement              19251 non-null  int64  
 13  ancien_code_

((19251, 40), None)

**19 251 lignes et 40 colonnes** : 22 colonnes numériques décimales (`float64`), 14 colonnes texte et 4 entières (`int64`).

Attention : une ligne n'est pas une vente. C'est un élément d'une vente (un lot, un local ou une parcelle). Le fichier ne contient que 7 465 ventes distinctes (`id_mutation`).

## Quelles colonnes ont beaucoup de valeurs manquantes ? Pourquoi, à ton avis ?

In [2]:
# En pourcentage :
print((df.isnull().mean() * 100).sort_values(ascending=False))

lot4_surface_carrez             100.000000
ancien_id_parcelle              100.000000
ancien_code_commune             100.000000
lot5_surface_carrez             100.000000
ancien_nom_commune              100.000000
lot3_surface_carrez              99.989611
lot5_numero                      99.948055
lot2_surface_carrez              99.942860
lot4_numero                      99.844164
numero_volume                    99.553270
lot3_numero                      99.516908
lot1_surface_carrez              98.929926
lot2_numero                      98.644226
nature_culture_speciale          98.633837
code_nature_culture_speciale     98.633837
adresse_suffixe                  90.826451
lot1_numero                      66.266687
surface_reelle_bati              58.158018
surface_terrain                  34.595605
code_nature_culture              34.595605
nature_culture                   34.595605
nombre_pieces_principales        26.808997
type_local                       26.788219
code_type_l

Plusieurs colonnes sont entièrement vides : `lot4_surface_carrez`, `lot5_surface_carrez`, `ancien_id_parcelle`, `ancien_code_commune` et `ancien_nom_commune`.

- `surface_reelle_bati` est vide pour les terrains et les dépendances, puisqu'il n'y a pas de bâti.
- `surface_terrain` est vide pour les appartements.
- `type_local` à NaN correspond à des lignes qui ne décrivent qu'une parcelle de terrain.
- Les colonnes `ancien_*` sont vides parce qu'aucune commune n'a fusionné à La Réunion.
- Les colonnes `lotN_*` sont presque vides : peu de ventes comportent plusieurs lots de copropriété.

## Quelles valeurs prennent nature_mutation et type_local ?

In [3]:
print("Pour la colonne 'nature_mutation', voici le nombre de valeurs manquantes et les différentes valeurs présentes :")
print(df['nature_mutation'].value_counts(dropna=False))

print("\nPour la colonne 'type_local', voici le nombre de valeurs manquantes et les différentes valeurs présentes :")
print(df['type_local'].value_counts(dropna=False))

Pour la colonne 'nature_mutation', voici le nombre de valeurs manquantes et les différentes valeurs présentes :
nature_mutation
Vente                                 17542
Vente en l'état futur d'achèvement     1319
Vente terrain à bâtir                   202
Echange                                 118
Adjudication                             70
Name: count, dtype: int64

Pour la colonne 'type_local', voici le nombre de valeurs manquantes et les différentes valeurs présentes :
type_local
Dépendance                                  6003
NaN                                         5157
Appartement                                 4337
Maison                                      3228
Local industriel. commercial ou assimilé     526
Name: count, dtype: int64


- `nature_mutation` : surtout des ventes classiques (17 542 lignes), puis des ventes sur plan (VEFA), des terrains à bâtir, des échanges et des adjudications. Pour mesurer un prix de marché, on ne garde que les `Vente`.
- `type_local` : Dépendance, Appartement, Maison, Local industriel/commercial, et 5 157 lignes vides (les parcelles de terrain seules).

## Compte les lignes par id_mutation. Pourquoi une même vente apparaît-elle sur plusieurs lignes ? Que se passe-t-il si tu fais un mean() naïf de valeur_fonciere ? (C’est LE piège de DVF, prends le temps de le comprendre.)

In [4]:
# Compter combien de fois chaque id_mutation apparaît
counts = df['id_mutation'].value_counts()
print(counts.head(10))

print("\nExemple de mutation avec plusieurs lignes :")

# Regarder un exemple de mutation qui a plusieurs lignes
# Afficher les lignes pour cette mutation précise
exemple = df[df['id_mutation'] == '2024-1185753']
print(exemple[['id_mutation', 'valeur_fonciere', 'lot1_numero', 'surface_reelle_bati', 'type_local']])

id_mutation
2024-1188637    3048
2024-1187596     303
2024-1186245     220
2024-1189429     122
2024-1187895     103
2024-1186146     102
2024-1189612      85
2024-1189616      79
2024-1192287      72
2024-1185970      69
Name: count, dtype: int64

Exemple de mutation avec plusieurs lignes :
    id_mutation  valeur_fonciere lot1_numero  surface_reelle_bati   type_local
0  2024-1185753         197000.0         144                  NaN   Dépendance
1  2024-1185753         197000.0          21                  NaN   Dépendance
2  2024-1185753         197000.0          18                 65.0  Appartement


In [5]:
# Le piège : valeur_fonciere est répétée sur chaque ligne de la vente
print(f"Moyenne naïve (par ligne)     : {df['valeur_fonciere'].mean():>12,.0f} €")

ventes_uniques = df.drop_duplicates('id_mutation')
print(f"Moyenne par vente (dédoublée) : {ventes_uniques['valeur_fonciere'].mean():>12,.0f} €")
print(f"Médiane par vente             : {ventes_uniques['valeur_fonciere'].median():>12,.0f} €")

Moyenne naïve (par ligne)     :    2,663,787 €
Moyenne par vente (dédoublée) :      273,114 €
Médiane par vente             :      157,600 €


Une vente apparaît sur plusieurs lignes parce que DVF décrit **chaque bien de la vente séparément** : un appartement, sa cave, son parking, chaque parcelle… Toutes ces lignes portent le même `id_mutation` et **la même `valeur_fonciere`**, qui est le prix total de la vente.

Un `mean()` naïf compte donc le prix d'une vente autant de fois qu'elle a de lignes. La vente 2024-1188637 à Saint-Paul (10,8 M€, 1 008 appartements et 2 040 dépendances) pèse à elle seule 3 048 fois. Résultat : la moyenne naïve est d'environ 2,66 M€, contre 273 000 € une fois les ventes dédoublées. Il faut toujours raisonner par `id_mutation`.

## Quelles valeurs de valeur_fonciere sont aberrantes (1 €, plusieurs dizaines de millions) ?

In [6]:
print(df['valeur_fonciere'].describe())

# Ventes symboliques (≤ 1 €)
print("\nNombre de ventes à 1 € :", df.loc[df['valeur_fonciere'] <= 1, 'id_mutation'].nunique())

# Ventes ≥ 10 M€ : combien de lignes chacune ?
grosses = df[df['valeur_fonciere'] >= 10_000_000]
print(grosses.groupby(['id_mutation', 'nom_commune', 'valeur_fonciere']).size().rename('nb_lignes'))

count    1.920300e+04
mean     2.663787e+06
std      4.476859e+06
min      1.000000e+00
25%      1.180000e+05
50%      2.550000e+05
75%      2.223000e+06
max      2.378320e+07
Name: valeur_fonciere, dtype: float64

Nombre de ventes à 1 € : 48
id_mutation   nom_commune    valeur_fonciere
2024-1186245  La Possession  12871500.0          220
2024-1188356  Sainte-Marie   16352337.0            1
2024-1188637  Saint-Paul     10814215.0         3048
2024-1189038  L'Étang-Salé   15850000.0            2
2024-1189411  Le Tampon      14413062.0            4
2024-1189429  La Possession  21247446.0          122
2024-1190174  Le Tampon      12419625.0            2
2024-1190466  Le Tampon      10121224.0           46
2024-1190924  Les Avirons    20162674.0           15
2024-1191439  Saint-Denis    11935000.0            2
2024-1191443  Saint-Denis    13743912.0            5
2024-1191477  Saint-Denis    23783200.0            4
2024-1192461  Saint-Benoît   21134948.0           22
2024-1192609  Saint-Den

Deux types de valeurs aberrantes :

- **Les ventes à 1 €** (48 ventes) : ce sont des cessions symboliques, par exemple entre une commune et un bailleur ou au sein d'une famille. Ce ne sont pas des prix de marché.
- **Les ventes à plus de 10 M€** (14 ventes) : ce ne sont presque jamais des maisons hors de prix, mais surtout des **ventes en bloc** (immeubles, terrains d'opérations, locaux d'entreprise). La vente de La Possession à 12,9 M€ s'étale sur 220 lignes, celle de Saint-Paul à 10,8 M€ sur 3 048 lignes. On retrouve le piège précédent : un seul prix pour des dizaines, voire des milliers de lots.

Ces deux cas justifient de ne garder que les ventes d'un seul bien et d'exclure les prix trop bas.

## Calcule un prix médian au m² pour les appartements et les maisons vendus seuls, par commune. Quelles communes sont les plus chères ?

In [7]:
def ventes_simples(df, type_local):
    # Vraies ventes uniquement
    ventes = df[df['nature_mutation'] == 'Vente'].copy()
    
    # Mutations avec un seul local principal
    principaux = ventes[ventes['type_local'].isin(['Maison', 'Appartement', 'Local industriel. commercial ou assimilé'])]
    nb_principaux = principaux.groupby('id_mutation').size()
    mutations_simples = nb_principaux[nb_principaux == 1].index
    
    # Filtrer pour le type de local souhaité
    locaux = ventes[
        (ventes['id_mutation'].isin(mutations_simples)) &
        (ventes['type_local'] == type_local) &
        (ventes['surface_reelle_bati'] > 0) &
        (ventes['valeur_fonciere'] > 1000)   # exclut les ventes symboliques à 1 €
    ].copy()

    locaux['prix_m2'] = locaux['valeur_fonciere'] / locaux['surface_reelle_bati']
    locaux['hors_marche'] = locaux['prix_m2'] < 500

    return locaux          # une ligne par vente, avec prix_m2


def prix_median_m2(df, type_local):
    locaux = ventes_simples(df, type_local)
    locaux = locaux[~locaux['hors_marche']]   # exclusion ICI, au moment de l'agrégation
    return locaux.groupby('nom_commune')['prix_m2'].agg(['median', 'count']).sort_values('median', ascending=False)

cols = ['id_mutation', 'nom_commune', 'code_postal', 'valeur_fonciere',
        'surface_reelle_bati', 'surface_terrain', 'nombre_pieces_principales', 'prix_m2', 'hors_marche']
    
appart = ventes_simples(df, 'Appartement')
display(appart['prix_m2'].describe())
display(appart.nsmallest(5, 'prix_m2')[cols])
display(appart.nlargest(5, 'prix_m2')[cols])

maison = ventes_simples(df, 'Maison')
display(maison['prix_m2'].describe())
display(maison.nsmallest(5, 'prix_m2')[cols])
display(maison.nlargest(5, 'prix_m2')[cols])

count     2003.000000
mean      3028.829390
std       1429.380890
min         86.250000
25%       2170.271514
50%       2710.000000
75%       3428.571429
max      15441.176471
Name: prix_m2, dtype: float64

,id_mutation,nom_commune,code_postal,valeur_fonciere,surface_reelle_bati,surface_terrain,nombre_pieces_principales,prix_m2,hors_marche
16120,2024-1191828,Saint-Joseph,97480.0,6900.00,80.0,4650.0,2.0,86.250000,True
12444,2024-1189951,Saint-Denis,97490.0,8000.00,42.0,NaN,2.0,190.476190,True
13708,2024-1190574,Saint-Pierre,97410.0,48752.75,164.0,NaN,4.0,297.272866,True
5516,2024-1188118,Saint-André,97440.0,35000.00,55.0,142.0,3.0,636.363636,False
4826,2024-1187819,Saint-Denis,97490.0,54000.00,80.0,NaN,3.0,675.000000,False


,id_mutation,nom_commune,code_postal,valeur_fonciere,surface_reelle_bati,surface_terrain,nombre_pieces_principales,prix_m2,hors_marche
650,2024-1186076,Saint-Paul,97434.0,262500.0,17.0,NaN,1.0,15441.176471,False
4877,2024-1187851,Saint-Paul,97434.0,540000.0,39.0,NaN,2.0,13846.153846,False
5199,2024-1187955,Saint-Paul,97434.0,486000.0,36.0,NaN,3.0,13500.000000,False
5823,2024-1188290,Saint-Paul,97434.0,494680.0,37.0,NaN,2.0,13369.729730,False
18651,2024-1192975,Saint-Paul,97434.0,1100000.0,84.0,NaN,1.0,13095.238095,False


count     2171.000000
mean      3103.654839
std       2105.520219
min         20.000000
25%       1837.579633
50%       2765.957447
75%       3846.153846
max      30000.000000
Name: prix_m2, dtype: float64

,id_mutation,nom_commune,code_postal,valeur_fonciere,surface_reelle_bati,surface_terrain,nombre_pieces_principales,prix_m2,hors_marche
6455,2024-1188616,Saint-Leu,97416.0,3000.0,150.0,NaN,4.0,20.000000,True
10875,2024-1189310,Les Avirons,97425.0,3000.0,130.0,1246.0,5.0,23.076923,True
5498,2024-1188109,Saint-Benoît,97470.0,2000.0,80.0,602.0,5.0,25.000000,True
1018,2024-1186225,Sainte-Marie,97438.0,3000.0,109.0,1092.0,4.0,27.522936,True
10545,2024-1189144,Entre-Deux,97414.0,2175.0,66.0,15.0,5.0,32.954545,True


,id_mutation,nom_commune,code_postal,valeur_fonciere,surface_reelle_bati,surface_terrain,nombre_pieces_principales,prix_m2,hors_marche
6377,2024-1188572,Saint-Paul,97434.0,1500000.0,50.0,237.0,3.0,30000.000000,False
15928,2024-1191727,Saint-Paul,97434.0,2381620.0,117.0,863.0,4.0,20355.726496,False
18727,2024-1193014,Les Trois-Bassins,97426.0,2105000.0,113.0,1068.0,4.0,18628.318584,False
13059,2024-1190277,Saint-Paul,97434.0,1950000.0,112.0,713.0,3.0,17410.714286,False
1927,2024-1186583,Saint-Paul,97434.0,1703286.0,105.0,818.0,4.0,16221.771429,False


In [8]:
print("--- Prix médian au m² : APPARTEMENTS ---")
display(prix_median_m2(df, 'Appartement').head(5))

print("--- Prix médian au m² : MAISONS ---")
display(prix_median_m2(df, 'Maison').head(5))

--- Prix médian au m² : APPARTEMENTS ---


,median,count
nom_commune,,
Saint-Paul,5108.747045,224
Saint-Leu,4080.603448,59
Saint-Pierre,3263.760897,220
Les Avirons,3214.285714,25
La Possession,2838.775510,113


--- Prix médian au m² : MAISONS ---


,median,count
nom_commune,,
L'Étang-Salé,4851.091954,38
Saint-Leu,4288.841567,90
Saint-Paul,4029.850746,245
La Possession,3612.288106,93
Saint-Denis,3301.658449,268


La méthode : uniquement des `Vente`, avec un seul local principal, un prix supérieur à 1 000 € et sans les ventes hors marché (< 500 €/m²).

- **Appartements** : Saint-Paul est loin devant (≈ 5 100 €/m²), puis Saint-Leu (≈ 4 100 €/m²), Saint-Pierre et Les Avirons (≈ 3 200-3 300 €/m²).
- **Maisons** : L'Étang-Salé arrive en tête (≈ 4 850 €/m²), devant Saint-Leu (≈ 4 300 €/m²), Saint-Paul (≈ 4 000 €/m²) et La Possession (≈ 3 600 €/m²).

Ce sont surtout les communes de l'Ouest, côté plages : la côte « sous le vent ». Attention aux petits effectifs : Les Avirons (25 ventes) ou L'Étang-Salé (38 ventes) donnent des médianes moins fiables que Saint-Paul (plus de 200 ventes).

### Problèmes de qualité identifiés



In [9]:
print(appart['hors_marche'].sum())
print(maison['hors_marche'].sum())

3
80


Ventes au prix au m² < 500 € (`hors_marche`) : **3** appartements sur 2 003 et **80** maisons sur 2 171. Elles sont gardées dans `ventes_simples`, mais exclues au moment du calcul des médianes.

Récapitulatif des problèmes rencontrés :
- une même vente répartie sur plusieurs lignes, avec le prix répété (le piège `id_mutation`) ;
- des prix symboliques (1 €) et des ventes en bloc de plusieurs millions ;
- des ventes hors marché à quelques centaines d'euros le m² ;
- des colonnes vides ou presque vides, à ignorer.

### Conclusion

- **Pièges DVF** : une vente = plusieurs lignes (lots, caves, parcelles) qui répètent toutes la même `valeur_fonciere` ; un `mean()` naïf compte le prix plusieurs fois. Il faut raisonner par `id_mutation` et ne garder que les ventes avec un seul local principal.
- **Ventes hors marché** : on trouve des prix à 1 € ou quelques milliers d'euros (donations déguisées, ventes entre proches, bâti en ruine). Le seuil de 500 €/m² en écarte 3 appartements et 80 maisons ; il est arbitraire mais reste bien en dessous des prix réels de l'île (médiane ≈ 2 700 €/m²).
- **Cas du HLM** : l'appartement de Saint-Pierre (164 m², 4 pièces, 48 752,75 €, soit ≈ 297 €/m²) a un prix au centime près, ce qui ressemble à une cession d'un bailleur social à son locataire avec décote. C'est une vraie vente, mais pas un prix de marché : le seuil l'exclut à raison.
- **Saint-Gilles et la limite « bâti sans terrain »** : les prix records sont à Saint-Paul 97434 (Saint-Gilles), jusqu'à 30 000 €/m² pour une maison de 50 m² sur 237 m² de terrain. Le prix au m² divise toute la valeur par la seule surface bâtie, donc le prix du terrain (très cher près du lagon) gonfle l'indicateur. Pour les maisons, il faudrait tenir compte de `surface_terrain`.